# Experimental Design Participant Statistics

This notebook reproduces the participant statistics reported in the main paper's Experimental Design section. It builds the course-by-condition table from the valid-user roster and platform behavior-log timestamps, then runs the Welch baseline check for prior-knowledge scores mentioned in the text.


## Setup

The setup resolves repository-relative paths so the notebook works from Jupyter with the repository Python environment.

In [1]:
from pathlib import Path
import sys


def _find_repo_root():
    """Find the repository root from either Jupyter or the test runner."""
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "analyze").exists():
            return candidate
    raise RuntimeError("Could not locate repository root.")


def _data_path(*candidates):
    """Return the first existing data path among repository-relative aliases."""
    for candidate in candidates:
        path = Path(candidate)
        if not path.is_absolute():
            path = REPO_ROOT / path
        if path.exists():
            return path
    checked = ", ".join(str(c) for c in candidates)
    raise FileNotFoundError(f"None of these data paths exist: {checked}")


REPO_ROOT = _find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("Repository:", ".")


Repository: .


In [2]:
from dataclasses import dataclass
import json
import sqlite3
from typing import Dict, Iterable, List, Optional

import pandas as pd

from analyze.config.paths import BEHAVIORS_DB, CAPTEST_SCORE_FILE, EXCLUDE_USER_FILE, VALIDUSER_FILE
from analyze.core.data_processing import load_capability_scores, load_valid_users
from analyze.utils.display import relative_path


## Helper Functions

Duration is computed as the sum of within-session first-to-last action windows for each participant. Sessions are split after long inactivity gaps and at administrator progress events, matching the original summary script.

In [3]:
@dataclass(frozen=True)
class GroupStats:
    course: str
    group: str
    n_total: int
    prior_mean: Optional[float]
    prior_std: Optional[float]
    prior_n: int
    duration_mean: Optional[float]
    duration_std: Optional[float]
    duration_n: int


def _load_overall_durations(db_path: Path, session_gap_hours: float, split_on_admin_progress: bool) -> Dict[str, float]:
    """Return per-user total duration in minutes, summed across sessions."""
    if not db_path.exists():
        print(f"Behavior DB not found: {relative_path(db_path)}")
        return {}

    con = sqlite3.connect(str(db_path))
    cur = con.cursor()
    cur.execute("SELECT username, timestamp, action, value FROM user_actions WHERE deleted=0")

    session_gap_sec = session_gap_hours * 3600.0
    per_user_rows: Dict[str, List[tuple[float, str, Optional[str]]]] = {}
    for username, ts, action, value in cur.fetchall():
        if not isinstance(username, str):
            continue
        try:
            ts_f = float(ts)
        except Exception:
            continue
        per_user_rows.setdefault(username, []).append((ts_f, action or "", value))
    con.close()

    durations: Dict[str, float] = {}
    for username, rows in per_user_rows.items():
        rows.sort(key=lambda r: r[0])
        total_seconds = 0.0
        session_start = rows[0][0]
        prev_ts = rows[0][0]

        for ts_f, action, value in rows[1:]:
            gap = ts_f - prev_ts
            split = session_gap_hours > 0 and gap >= session_gap_sec

            if split_on_admin_progress and action == "progress" and value:
                try:
                    payload = json.loads(value)
                except Exception:
                    payload = {}
                if isinstance(payload, dict) and payload.get("admin") is True:
                    split = True

            if split:
                total_seconds += max(0.0, prev_ts - session_start)
                session_start = ts_f

            prev_ts = ts_f

        total_seconds += max(0.0, prev_ts - session_start)
        durations[username] = total_seconds / 60.0

    return durations


def _apply_overall_duration(df: pd.DataFrame, durations_by_user: Dict[str, float], cap_minutes: Optional[float]) -> pd.DataFrame:
    def _get_duration(username: str) -> Optional[float]:
        value = durations_by_user.get(username)
        if value is None:
            return None
        if cap_minutes is not None:
            return min(cap_minutes, value)
        return value

    df = df.copy()
    df["duration_min"] = df["username"].apply(_get_duration)
    return df


def _safe_mean_std(values: Iterable[float]) -> tuple[Optional[float], Optional[float], int]:
    series = pd.Series([v for v in values if v is not None and not pd.isna(v)])
    if series.empty:
        return None, None, 0
    return float(series.mean()), float(series.std(ddof=1)), int(series.count())


def _summarize_group(df: pd.DataFrame, course: str, group_label: str) -> GroupStats:
    prior_mean, prior_std, prior_n = _safe_mean_std(df["captest_score"].tolist())
    duration_mean, duration_std, duration_n = _safe_mean_std(df["duration_min"].tolist())
    return GroupStats(course, group_label, len(df), prior_mean, prior_std, prior_n, duration_mean, duration_std, duration_n)


def _build_summary_rows(df: pd.DataFrame, course_label: str) -> List[GroupStats]:
    return [
        _summarize_group(df[df["group"] == 1], course_label, "Experimental"),
        _summarize_group(df[df["group"] == 0], course_label, "Control"),
        _summarize_group(df, course_label, "Subtotal"),
    ]


def _format_float(value: Optional[float]) -> str:
    return "NA" if value is None else f"{value:.3f}"


## Load Participant Data

The roster provides randomized condition assignment. Prior-knowledge scores come from the capability test file, and durations come from the merged platform behavior database.

In [4]:
# Parameters from the original main-paper statistics script.
CAP_MINUTES = 0.0
SESSION_GAP_HOURS = 6.0
SPLIT_ON_ADMIN_PROGRESS = True

cap_minutes = None if CAP_MINUTES <= 0 else CAP_MINUTES

import contextlib
import io

with contextlib.redirect_stdout(io.StringIO()):
    python_df, math_df = load_valid_users(VALIDUSER_FILE, EXCLUDE_USER_FILE)
    py_captest, math_captest = load_capability_scores(CAPTEST_SCORE_FILE, EXCLUDE_USER_FILE)

python_df = python_df.merge(py_captest, on="username", how="left")
math_df = math_df.merge(math_captest, on="username", how="left")
python_df = python_df.copy()
math_df = math_df.copy()
python_df["course"] = "Python"
math_df["course"] = "Game Theory"

durations_by_user = _load_overall_durations(
    BEHAVIORS_DB,
    session_gap_hours=SESSION_GAP_HOURS,
    split_on_admin_progress=SPLIT_ON_ADMIN_PROGRESS,
)
python_df = _apply_overall_duration(python_df, durations_by_user, cap_minutes)
math_df = _apply_overall_duration(math_df, durations_by_user, cap_minutes)
overall_df = pd.concat([python_df, math_df], ignore_index=True)


## Participant Statistics Table

This table matches the columns reported in `tab:stat`: course, condition, intervention, sample size, and average duration.


In [5]:
rows: List[GroupStats] = []
rows.extend(_build_summary_rows(math_df, "Game Theory"))
rows.extend(_build_summary_rows(python_df, "Python"))
rows.append(_summarize_group(overall_df, "Overall", "All"))

def _intervention(group: str) -> str:
    if group == "Experimental":
        return "GPT-assisted"
    if group == "Control":
        return "No LLM allowed"
    return "-"


table_df = pd.DataFrame(
    [
        {
            "Course": r.course,
            "Condition": r.group,
            "Intervention": _intervention(r.group),
            "Sample Size": r.n_total,
            "Avg. Duration (min)": f"{_format_float(r.duration_mean)} +/- {_format_float(r.duration_std)}",
        }
        for r in rows
    ]
)

table_df


,Course,Condition,Intervention,Sample Size,Avg. Duration (min)
0,Game Theory,Experimental,GPT-assisted,158,102.339 +/- 16.949
1,Game Theory,Control,No LLM allowed,79,96.711 +/- 20.694
2,Game Theory,Subtotal,-,237,100.463 +/- 18.431
3,Python,Experimental,GPT-assisted,54,103.537 +/- 21.418
4,Python,Control,No LLM allowed,27,104.959 +/- 17.320
5,Python,Subtotal,-,81,104.011 +/- 20.046
6,Overall,All,-,318,101.367 +/- 18.887


## Prior-Knowledge Balance Check

The Experimental Design text reports Welch's t-tests showing no significant prior-knowledge difference between Experimental and Control groups within either course.


In [6]:
def _clean_scores(df: pd.DataFrame) -> List[float]:
    return [float(v) for v in df["captest_score"].tolist() if v is not None and not pd.isna(v)]


def _welch_p_value(exp_vals: List[float], ctrl_vals: List[float]) -> Optional[float]:
    try:
        from scipy.stats import ttest_ind  # type: ignore
    except Exception:
        return None
    if not exp_vals or not ctrl_vals:
        return None
    _t_stat, p_value = ttest_ind(exp_vals, ctrl_vals, equal_var=False, nan_policy="omit")
    return float(p_value)


balance_rows = []
for label, df in [("Python", python_df), ("Game Theory", math_df)]:
    exp_vals = _clean_scores(df[df["group"] == 1])
    ctrl_vals = _clean_scores(df[df["group"] == 0])
    p_value = _welch_p_value(exp_vals, ctrl_vals)
    balance_rows.append({"Course": label, "Welch p-value": p_value})

balance_df = pd.DataFrame(balance_rows)
balance_df.round(4)


,Course,Welch p-value
0,Python,0.2584
1,Game Theory,0.7636
